In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
print(Path.cwd())

/Users/majdalsuleh/Projects/mining-quality-prediction


In [2]:
import duckdb

con = duckdb.connect("data/mining.duckdb")

con.execute("""
    CREATE OR REPLACE TABLE raw AS
    SELECT *
    FROM read_csv('data/MiningProcess_Flotation_Plant_Database.csv',
                  header = true,
                  decimal_separator = ',')
""")

con.sql("DESCRIBE raw")

┌──────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│         column_name          │ column_type │  null   │   key   │ default │  extra  │
│           varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ date                         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ % Iron Feed                  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ % Silica Feed                │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Starch Flow                  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Amina Flow                   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Ore Pulp Flow                │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Ore Pulp pH                  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Ore Pulp Density             │ DOUBLE    

In [3]:
con.sql("SELECT COUNT(*) AS rows, MIN(date), MAX(date) FROM raw")

┌────────┬─────────────────────┬─────────────────────┐
│  rows  │      min(date)      │      max(date)      │
│ int64  │      timestamp      │      timestamp      │
├────────┼─────────────────────┼─────────────────────┤
│ 737453 │ 2017-03-10 01:00:00 │ 2017-09-09 23:00:00 │
└────────┴─────────────────────┴─────────────────────┘

In [4]:
con.execute(open("sql/01_hourly_table.sql").read())
con.sql("SELECT COUNT(*) AS hours, SUM(is_filled::INT) AS filled_hours FROM hourly")

┌───────┬──────────────┐
│ hours │ filled_hours │
│ int64 │    int128    │
├───────┼──────────────┤
│  4097 │          448 │
└───────┴──────────────┘

In [5]:
con.sql("SELECT * FROM hourly LIMIT 5")

┌─────────────────────┬────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬───────────┐
│        hour         │ n_readings │     iron_feed      │    silica_feed     │    starch_flow     │    amina_flow     │     pulp_flow      │      pulp_ph       │    pulp_density    │       air_01       │       air_02       │       air_03       │       air_04       │       air_05       │       air_06       │       air_07       │      level_01      │      level_02      │      level_03      │     level_04      │      level_05      │      level_06      │      level_07   